# Fabric Try-On GPU Worker (CatVTON) on Colab
Run cells top to bottom. Cell 1 sets up the environment. Cell 2 starts the server.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────

# 1a. Mount Google Drive to cache HuggingFace model weights (~6-8 GB)
#     After first run, this skips the download and loads from Drive.
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'

# 1b. Clone your repo
!git clone https://github.com/S-M-Sohel1/try-on-gpu.git
%cd try-on-gpu/gpu_worker

# 1c. Install Python dependencies
!pip install -q -r requirements.txt
!pip install -q nest_asyncio

# 1d. Clone CatVTON (provides model.pipeline, model.cloth_masker, etc.)
!git clone https://github.com/Zheng-Chong/CatVTON.git
import sys
sys.path.append(os.path.abspath('CatVTON'))

# 1e. Install Detectron2 — required by CatVTON's DensePose AutoMasker
#     This gives the best automatic masking (no gloves, no artifacts)
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

print('\n✅ Environment ready!')

In [ ]:
# ── Cell 2: Start Server ──────────────────────────────────────────────────────
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# IMPORTANT: Replace with your ngrok authtoken from https://dashboard.ngrok.com
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN_HERE")

public_url = ngrok.connect(8000)
print('Public URL:', public_url)
print('Paste this URL into test_infer.py on your local machine.')

# Start FastAPI server (models load during startup)
config = uvicorn.Config('main:app', host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()